# Linear elasticity: material robustness

2D cantilevered rectangular beam under gravity. Fixed on the left face; traction-free on the right, top, and bottom.

Two Hellinger–Reissner mixed schemes are implemented and can be selected at runtime:
- **JM** (Johnson–Mercier): symmetric $P_1$ stress in $H(\operatorname{div})$, $P_0$ displacement.
- **AFW** (Arnold–Falk–Winther): $[BDM_1]^{2\times 2}$ stress with weakly imposed symmetry, $P_0$ displacement, scalar $DG_0$ skew-rotation variable $\gamma$.

In [162]:
try:
    !wget "https://fem-on-colab.github.io/releases/firedrake-install-development-real.sh" -O "/tmp/firedrake-install.sh"
    !bash "/tmp/firedrake-install.sh"
    from firedrake import *  # noqa: F401
except:
    from firedrake import *  # noqa: F401

--2026-05-28 02:39:12--  https://fem-on-colab.github.io/releases/firedrake-install-development-real.sh
Resolving fem-on-colab.github.io (fem-on-colab.github.io)... 185.199.108.153, 185.199.109.153, 185.199.110.153, ...
Connecting to fem-on-colab.github.io (fem-on-colab.github.io)|185.199.108.153|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4775 (4.7K) [application/x-sh]
Saving to: ‘/tmp/firedrake-install.sh’

/tmp/firedrake-inst 100%[===================>]   4.66K  --.-KB/s    in 0s      

2026-05-28 02:39:12 (47.5 MB/s) - ‘/tmp/firedrake-install.sh’ saved [4775/4775]

+ INSTALL_PREFIX=/usr/local
++ echo /usr/local
++ awk -F/ '{print NF-1}'
+ INSTALL_PREFIX_DEPTH=2
+ PROJECT_NAME=fem-on-colab
+ SHARE_PREFIX=/usr/local/share/fem-on-colab
+ FIREDRAKE_INSTALLED=/usr/local/share/fem-on-colab/firedrake.installed
+ [[ ! -f /usr/local/share/fem-on-colab/firedrake.installed ]]
+ set +x
























#########################################################

## Johnson–Mercier (JM)

Stress $\sigma_h$ in the Johnson–Mercier element — piecewise linear **symmetric** tensors in $H(\operatorname{div})$ — and displacement $u_h \in [P_0]^2$. Symmetry is built into the element.

Find $(\sigma_h, u_h) \in \Sigma_h^{JM} \times V_h$ such that

$$\int_\Omega A\sigma_h:\tau_h + u_h \cdot \operatorname{div}(\tau_h) \, dx = \int_{\Gamma_D} g \cdot (\tau_h n) \, ds \qquad \forall \tau_h \in \Sigma_h^{JM},$$
$$\int_\Omega \operatorname{div}(\sigma_h) \cdot v_h \, dx = -\int_\Omega f \cdot v_h \, dx \qquad \forall v_h \in V_h,$$

where $A\sigma = \frac{1}{2\mu}\sigma - \frac{\lambda}{2\mu(2\mu + 2\lambda)}\operatorname{tr}(\sigma)I$ is the compliance tensor.

## Arnold–Falk–Winther (AFW)

Stress $\sigma_h \in [BDM_1]^2$ (each row in $BDM_1$, **not** intrinsically symmetric) with displacement $u_h \in [P_0]^2$ and scalar skew-rotation $\gamma_h \in P_0$. Symmetry of $\sigma_h$ is enforced weakly via $\gamma_h$.

Find $(\sigma_h, u_h, \gamma_h) \in \Sigma_h^{AFW} \times V_h \times Q_h$ such that

$$\int_\Omega A\sigma_h:\tau_h + u_h \cdot \operatorname{div}(\tau_h) + \gamma_h(\tau_{h,01} - \tau_{h,10}) \, dx = \int_{\Gamma_D} g \cdot (\tau_h n) \, ds \qquad \forall \tau_h,$$
$$\int_\Omega \operatorname{div}(\sigma_h) \cdot v_h \, dx = -\int_\Omega f \cdot v_h \, dx \qquad \forall v_h,$$
$$\int_\Omega (\sigma_{h,01} - \sigma_{h,10})\,\eta_h \, dx = 0 \qquad \forall \eta_h,$$

where $A$ acts on $\operatorname{sym}(\sigma_h)$. More unknowns than JM, but the stress space is simpler to construct.

## Code

In [ ]:
def solve_elasticity(scheme="JM", length=10.0, height=1.0, nu=0.3, nx=20, ny=20):
    """
    Solve 2D cantilevered beam under gravity.
    scheme : "JM" (Johnson-Mercier) or "AFW" (Arnold-Falk-Winther)
    Returns sigma_h, u_h, sig_vm, mesh.
    """
    # Mesh
    mesh = RectangleMesh(nx, ny, length, height)
    x, y = SpatialCoordinate(mesh)
    d = 2

    # Material parameters
    nu_c  = Constant(nu)
    mu    = E_c / (2*(1 + nu_c))
    lmbda = E_c * nu_c / ((1 + nu_c) * (1 - 2*nu_c))
    def A_comp(s):
        # Compliance tensor
        return (1/(2*mu))*sym(s) - (lmbda/(2*mu*(2*mu + d*lmbda)))*tr(s)*Identity(d)

    # Function spaces + functions
    DG_vec = VectorFunctionSpace(mesh, "DG", 0, variant="alfeld")
    if scheme == "JM":
        JM = FunctionSpace(mesh, "JM", 1)
        W  = JM * DG_vec
        sigma, u_0 = TrialFunctions(W)
        tau,   v = TestFunctions(W)
    elif scheme == "AFW":
        AFW_elem = VectorElement(FiniteElement("BDM", mesh.ufl_cell(), 1), dim=2)
        AFW = FunctionSpace(mesh, AFW_elem)
        DG = FunctionSpace(mesh, "DG", 0)
        W = AFW * DG_vec * DG
        sigma, u_0, gamma = TrialFunctions(W)
        tau,   v, eta   = TestFunctions(W)
        a = (
            inner(A_comp(sigma), tau)
          + inner(u_0, div(tau))
          + inner(gamma, asym2d(tau))
          + inner(div(sigma), v)
          + inner(asym2d(sigma), eta)
        ) * dx
    else:
        raise ValueError(f"Unknown scheme '{scheme}'. Choose 'JM' or 'AFW'.")
    
    # LHS
    a = (
        inner(A_comp(sigma), tau)
      + inner(u_0, div(tau))
      + inner(div(sigma), v)
    ) * dx
    if scheme == "AFW":
        def asym2d(tau): return tau[0, 1] - tau[1, 0]
        a += (
            inner(gamma, asym2d(tau))
          + inner(asym2d(sigma), eta)
        ) * dx

    # RHS
    x_var = variable(x)
    phi = - 0.5 * cos(pi * x_var / length)
    phi_prime = diff(phi, x_var)
    u_BC = as_vector([(y - height/2) * phi_prime, phi])
    L  = inner(sym(grad(u_BC)), tau) * dx

    # Boundary conditions
    bcs = [
        DirichletBC(W.sub(0), 0, 3),
        DirichletBC(W.sub(0), 0, 4)
    ]

    # Solve
    w = Function(W)
    solve(a == L, w, bcs=bcs)

    # Output
    sigma, u_0 = w.subfunctions

    s00 = sigma_h[0, 0]
    s11 = sigma_h[1, 1]
    s01 = (sigma_h[0, 1] + sigma_h[1, 0]) / 2
    sig_vm = sqrt(s00**2 - s00*s11 + s11**2 + 3*s01**2)
    sig_vm_dg = Function(FunctionSpace(mesh, "DG", 0), name="Von Mises stress (DG0)").project(sig_vm)
    sig_vm_cg = Function(FunctionSpace(mesh, "CG", 1), name="Von Mises stress (CG1)").project(sig_vm)

    u = Function(DG, name="u (Alfeld-split DG0)").project(u_BC + u_0)
    u_cg = Function(VectorFunctionSpace(mesh, "CG", 1), name="u (CG1)").project(u)

    return sigma, (u, u_cg), (sig_vm_dg, sig_vm_cg), mesh

In [ ]:
def plot_solution(u_cg, sig_vm_cg, sig_vm_dg, mesh):
    """Plot vertical displacement, von Mises stress, and deformed mesh."""
    import matplotlib.pyplot as plt
    import matplotlib.tri as mtri

    fig, axes = plt.subplots(1, 2, figsize=(12, 3))

    # Vertical displacement
    c0 = tripcolor(u_cg.sub(1), axes=axes[0], cmap="RdBu_r")
    plt.colorbar(c0, ax=axes[0], label=r"$u_y$")
    # axes[0].set_aspect("equal")
    axes[0].set_title("Vertical displacement")

    # Von Mises stress
    c1 = tripcolor(sig_vm_cg, axes=axes[1], cmap="inferno")
    plt.colorbar(c1, ax=axes[1], label=r"$\sigma_\mathrm{vM}$")
    # axes[1].set_aspect("equal")
    axes[1].set_title("Von Mises stress")

    plt.tight_layout()
    plt.show()

    # Deformation (coloured by von Mises stress)
    coords = mesh.coordinates.dat.data_ro
    cells  = mesh.coordinates.cell_node_map().values
    disp   = u_cg.dat.data_ro

    triang_ref = mtri.Triangulation(coords[:, 0], coords[:, 1], cells)
    triang_def = mtri.Triangulation(
        coords[:, 0] + scale * disp[:, 0],
        coords[:, 1] + scale * disp[:, 1],
        cells,
    )

    fig, ax = plt.subplots(figsize=(10, 3))
    tc = ax.tripcolor(triang_def, facecolors=sig_vm_dg.dat.data_ro, cmap="inferno")
    plt.colorbar(tc, ax=ax, label=r"$\sigma_\mathrm{vM}$")
    ax.triplot(triang_ref, color="k", linewidth=0.3, alpha=0.2, label="Original")
    ax.set_aspect("equal")
    ax.legend()
    ax.set_title(f"Deformed mesh (von Mises stress)")
    plt.tight_layout()
    plt.show()

In [ ]:
def boundary_forces(sigma, mesh):
    """Print resultant forces on the left (clamped) and right (free) faces."""
    n = FacetNormal(mesh)
    for label, marker in [("Left  (clamped)", 1), ("Right (free)   ", 2)]:
        Fx = assemble(dot(sigma, n)[0] * ds(marker))
        Fy = assemble(dot(sigma, n)[1] * ds(marker))
        print(f"{label}  Fx = {Fx:+.6g},  Fy = {Fy:+.6g}")

## Testing

In [ ]:
scheme = "JM"

sigma, (u, u_cg), (sig_vm_dg, sig_vm_cg), mesh = solve_elasticity("JM")
plot_solution(u_cg, sig_vm_cg, sig_vm_dg, mesh)
boundary_forces(sigma, mesh)

NameError: name 'RectangleMesh' is not defined